In [2]:
"""
Multicollinearity Reduction via Pearson Correlation Analysis for Hardness Dataset.

This script identifies pairs of highly correlated features (|r| >= threshold) 
and eliminates the feature with weaker correlation to the target variable (HV). 
The refined dataset with reduced redundancy is exported for ML benchmarking.
"""

import os
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# =============================================================================
# 1. Load Cleaned Dataset & Prepare Features
# =============================================================================

input_path = r"./data/c_cleaned_hv_with_features.csv"
output_path = r"./data/d_hv_with_corr_features.csv"  

df_filtered = pd.read_csv(input_path)

target_column = "HV"
drop_cols = ["FILE_NAME", target_column]
feature_cols = [c for c in df_filtered.columns if c not in drop_cols]

# =============================================================================
# 2. Pearson Correlation Selection (Threshold = 0.95)
# =============================================================================

corr_threshold = 0.95

print(f"[INFO] Initiating Correlation-based Feature Selection (Threshold = {corr_threshold})...")
print(f"{'Target Feature Pair (Collinear)':<45} | {'Correlation':>11} | {'Removed Feature':>25}")
print("-" * 88)

# Compute feature correlation matrix and target correlation
corr_matrix = df_filtered[feature_cols].corr().abs()
target_corr = df_filtered[feature_cols].apply(lambda x: x.corr(df_filtered[target_column])).abs()

removed_features = set()

# Iterate over upper triangle of correlation matrix
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        col1 = feature_cols[i]
        col2 = feature_cols[j]
        
        # Skip if either feature has already been removed
        if col1 in removed_features or col2 in removed_features:
            continue
            
        r_val = corr_matrix.loc[col1, col2]
        
        # If correlation exceeds threshold, remove the one with lower target correlation
        if r_val >= corr_threshold:
            if target_corr[col1] >= target_corr[col2]:
                to_remove = col2
                to_keep = col1
            else:
                to_remove = col1
                to_keep = col2
                
            removed_features.add(to_remove)
            pair_str = f"{col1} <-> {col2}"
            print(f"{pair_str:<45} | {r_val:>11.3f} | {to_remove:>25}")

selected_feature_cols = [c for c in feature_cols if c not in removed_features]

print(f"\n[INFO] Correlation Selection completed. Removed {len(removed_features)} redundant features.")
print(f"[INFO] Retained Features ({len(selected_feature_cols)}): {selected_feature_cols}\n")

# =============================================================================
# 3. Export Refined Dataset with Selected Features
# =============================================================================

export_cols = [col for col in df_filtered.columns if col in ["FILE_NAME"] + selected_feature_cols + [target_column]]

df_output = df_filtered[export_cols]
df_output.to_csv(output_path, index=False)

print(f"[INFO] Processed dataset successfully exported to: {output_path}")

[INFO] Initiating Correlation-based Feature Selection (Threshold = 0.95)...
Target Feature Pair (Collinear)               | Correlation |           Removed Feature
----------------------------------------------------------------------------------------
phase_count_density <-> phase_euler_number    |       0.976 |        phase_euler_number
glcm_homogeneity_std <-> lbp_entropy          |       0.955 |      glcm_homogeneity_std
glcm_energy_mean <-> glcm_ASM_mean            |       0.995 |          glcm_energy_mean
lbp_mean <-> lbp_entropy                      |       0.976 |                  lbp_mean
lbp_entropy <-> lbp_uniformity                |       0.990 |            lbp_uniformity
intensity_skewness <-> intensity_kurtosis     |       0.973 |        intensity_kurtosis

[INFO] Correlation Selection completed. Removed 6 redundant features.
[INFO] Retained Features (42): ['phase_area_fraction', 'phase_count_density', 'phase_mean_area', 'phase_std_area', 'phase_max_area', 'phase_mean_ecc